# Codveda Data Analytics Internship — Level 1 (Basic)

**Intern:** Abubakar Rabiu Salihawa  
**Internship:** Data Analysis Intern, Codveda Technologies  
**Intern ID:** CV/AI/82271  

**Selected tasks:** Task 1 — Data Cleaning and Preprocessing; Task 2 — Exploratory Data Analysis (EDA).

This section covers the cleaning steps, exploratory analysis, visualisations and brief explanations of the results using the supplied datasets.

## Task 1: Data Cleaning and Preprocessing

The stock-price dataset is used because it contains missing values in the `open`, `high`, and `low` columns, a date column that must be converted to a proper datetime type, and many stock symbols that should be standardized. The workflow also checks and removes duplicate rows even when none are found.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Keep charts simple and consistent with an ordinary analyst report.
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.grid': False,
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 10
})
import seaborn as sns

ROOT = Path.cwd().parent
DATA = ROOT / 'data'
OUT = ROOT / 'outputs' / 'level1'
OUT.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 20)

stock_raw = pd.read_csv(DATA / 'stock_prices.csv')
print('Original shape:', stock_raw.shape)
print('Duplicate rows:', stock_raw.duplicated().sum())
print('Missing values before cleaning:')
display(stock_raw.isna().sum().to_frame('missing_count'))
display(stock_raw.head())

Original shape: (497472, 7)


Duplicate rows: 0
Missing values before cleaning:


        missing_count
symbol              0
date                0
open               11
high                8
low                 8
close               0
volume              0

  symbol        date      open      high       low     close    volume
0    AAL  2014-01-02   25.0700   25.8200   25.0600   25.3600   8998943
1   AAPL  2014-01-02   79.3828   79.5756   78.8601   79.0185  58791957
2    AAP  2014-01-02  110.3600  111.8800  109.2900  109.7400    542711
3   ABBV  2014-01-02   52.1200   52.3300   51.5200   51.9800   4569061
4    ABC  2014-01-02   70.1100   70.2300   69.4800   69.8900   1148391

In [2]:
stock = stock_raw.copy()
stock.columns = stock.columns.str.strip().str.lower().str.replace(' ', '_')
stock['symbol'] = stock['symbol'].astype(str).str.strip().str.upper()
stock['date'] = pd.to_datetime(stock['date'], errors='coerce')

rows_before = len(stock)
stock = stock.drop_duplicates().copy()
duplicates_removed = rows_before - len(stock)
stock = stock.sort_values(['symbol', 'date']).reset_index(drop=True)

price_cols = ['open', 'high', 'low', 'close']
stock[price_cols] = stock.groupby('symbol')[price_cols].transform(
    lambda s: s.interpolate(method='linear', limit_direction='both')
)
for col in price_cols:
    stock[col] = stock[col].fillna(stock.groupby('symbol')[col].transform('median'))
    stock[col] = stock[col].fillna(stock[col].median())

stock = stock.dropna(subset=['symbol', 'date', 'close']).reset_index(drop=True)
stock['high'] = stock[['open', 'high', 'low', 'close']].max(axis=1)
stock['low'] = stock[['open', 'high', 'low', 'close']].min(axis=1)
stock['volume'] = pd.to_numeric(stock['volume'], errors='coerce').fillna(0).clip(lower=0).astype('int64')
stock.to_csv(OUT / 'cleaned_stock_prices.csv', index=False)

cleaning_summary = pd.DataFrame({
    'Measure': ['Rows before cleaning', 'Rows after cleaning', 'Duplicates removed', 'Missing values before', 'Missing values after'],
    'Value': [len(stock_raw), len(stock), duplicates_removed, int(stock_raw.isna().sum().sum()), int(stock.isna().sum().sum())]
})
display(cleaning_summary)
print('Data types after cleaning:')
display(stock.dtypes.to_frame('dtype'))
print('Saved:', OUT / 'cleaned_stock_prices.csv')

                 Measure   Value
0   Rows before cleaning  497472
1    Rows after cleaning  497472
2     Duplicates removed       0
3  Missing values before      27
4   Missing values after       0

Data types after cleaning:


                 dtype
symbol          object
date    datetime64[ns]
open           float64
high           float64
low            float64
close          float64
volume           int64

Saved: outputs/level1/cleaned_stock_prices.csv


### Cleaning result

The dataset is now analysis-ready: dates are stored consistently, stock symbols are standardized, missing price values are handled within each company’s time series, duplicate checking is completed, and numerical fields have valid data types. Interpolation was preferred over deleting rows because a missing daily price can usually be estimated from neighbouring trading days without discarding the entire record.

## Task 2: Exploratory Data Analysis (EDA)

The Iris dataset is used for EDA because it contains four continuous measurements and a clearly defined species category. This makes it suitable for summary statistics, distributions, outlier checks, scatter plots, and correlation analysis.

In [3]:
iris = pd.read_csv(DATA / 'iris.csv')
print('Shape:', iris.shape)
print('Missing values:', int(iris.isna().sum().sum()))
print('Duplicate rows:', int(iris.duplicated().sum()))
display(iris.head())

numeric_cols = iris.select_dtypes(include='number').columns.tolist()
summary_stats = iris[numeric_cols].agg(['mean','median','std','min','max']).T
summary_stats['mode'] = iris[numeric_cols].mode().iloc[0]
summary_stats = summary_stats[['mean','median','mode','std','min','max']].round(3)
summary_stats.to_csv(OUT / 'iris_summary_statistics.csv')
display(summary_stats)

Shape: (150, 5)
Missing values: 0
Duplicate rows: 3


   sepal_length  sepal_width  petal_length  petal_width species
0           5.1          3.5           1.4          0.2  setosa
1           4.9          3.0           1.4          0.2  setosa
2           4.7          3.2           1.3          0.2  setosa
3           4.6          3.1           1.5          0.2  setosa
4           5.0          3.6           1.4          0.2  setosa

               mean  median  mode    std  min  max
sepal_length  5.843    5.80   5.0  0.828  4.3  7.9
sepal_width   3.054    3.00   3.0  0.434  2.0  4.4
petal_length  3.759    4.35   1.5  1.764  1.0  6.9
petal_width   1.199    1.30   0.2  0.763  0.1  2.5

In [4]:
# Histograms
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.ravel(), numeric_cols):
    ax.hist(iris[col], bins=15, edgecolor='black', alpha=0.8)
    ax.set_title(f'Distribution of {col.replace("_", " ").title()}')
    ax.set_xlabel(col.replace('_', ' ').title())
    ax.set_ylabel('Frequency')
plt.tight_layout()
plt.savefig(OUT / 'iris_histograms.png', dpi=200, bbox_inches='tight')
plt.show()

In [5]:
# Boxplots by species
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, col in zip(axes.ravel(), numeric_cols):
    sns.boxplot(data=iris, x='species', y=col, ax=ax)
    ax.set_title(f'{col.replace("_", " ").title()} by Species')
    ax.set_xlabel('Species')
    ax.set_ylabel(col.replace('_', ' ').title())
plt.tight_layout()
plt.savefig(OUT / 'iris_boxplots.png', dpi=200, bbox_inches='tight')
plt.show()

In [6]:
plt.figure(figsize=(9, 6))
for species, group in iris.groupby('species'):
    plt.scatter(group['petal_length'], group['petal_width'], label=species, alpha=0.8)
plt.title('Petal Length versus Petal Width')
plt.xlabel('Petal Length (cm)')
plt.ylabel('Petal Width (cm)')
plt.legend(title='Species')
plt.tight_layout()
plt.savefig(OUT / 'iris_scatter_petal.png', dpi=200, bbox_inches='tight')
plt.show()

corr = iris[numeric_cols].corr().round(3)
corr.to_csv(OUT / 'iris_correlation_matrix.csv')
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1, square=True)
plt.title('Correlation Matrix of Iris Measurements')
plt.tight_layout()
plt.savefig(OUT / 'iris_correlation_heatmap.png', dpi=200, bbox_inches='tight')
plt.show()
display(corr)

              sepal_length  sepal_width  petal_length  petal_width
sepal_length         1.000       -0.109         0.872        0.818
sepal_width         -0.109        1.000        -0.421       -0.357
petal_length         0.872       -0.421         1.000        0.963
petal_width          0.818       -0.357         0.963        1.000

### Main EDA findings

1. Petal length and petal width have a very strong positive relationship, so flowers with longer petals generally also have wider petals.
2. Setosa is clearly separated from the other species by its much smaller petal measurements.
3. Versicolor and virginica overlap more in sepal measurements, but petal measurements distinguish them more clearly.
4. Sepal width has a weaker relationship with the other numerical variables, so it is less useful as a single distinguishing feature.
5. The duplicate check identifies repeated observations, but they are not automatically removed in EDA because identical flower measurements can be valid observations rather than data-entry errors.

## Level 1 conclusion

For Level 1, I cleaned the stock-price data and carried out exploratory analysis on the Iris data. The saved outputs show the cleaned dataset, summary statistics, correlations, and the required charts.